# Topics covered in the previous lecture

-   Linear models:

    -   Relationship between $y$ and $\mathbf{x}$ given by:
        $$
            y_i = \mu + \mathbf{x}_i'\bm{\beta} + \epsilon_i
        $$

    -   Estimated by minimizing the **loss function** (OLS):
        $$
        L(\mu, \bm{\beta}) = 
            \underbrace{\sum_{i=1}^N \Bigl(
            y_i - \mu - \mathbf{x}_i'\bm{\beta}\Bigr)^2}_{\text{Sum of squared errors}}
        $$
-   Creating additional features using polynomials
-   Cross-validation to determine hyperparameters (polynomial degree)

***
# This week

Linear regression models with **regularization**:

-   Ridge regression
-   Lasso
-   Elastic Net (lecture notes)

***

# Ridge regression

-   Loss function:
    $$
    L(\mu, \bm{\beta}) = 
        \underbrace{\sum_{i=1}^N \Bigl(
        y_i - \mu - \mathbf{x}_i'\bm{\beta}\Bigr)^2}_{\text{Sum of squared errors}}
        + 
        \underbrace{\alpha \sum_{k=1}^K\beta_k^2}_{\text{L2 penalty}}
    $$

-   Penalty term introduces **shrinkage** or **regularization**

    -   Large coefficients $\beta_k$ are penalized

-   The resulting model is **biased**, but has **lower variance** when making predictions
    on new data

***
## Example: Polynomial approximation

-   The true relationship is given by a trigonometric function, with error $\epsilon_i$:
    $$
    \begin{aligned}
    y_i &= \cos\left( \frac{3\pi}{200} x_i \right) + \epsilon_i \\
        \epsilon_i &\stackrel{\text{iid}}{\sim} \mathcal{N}\left(0, 0.5^2\right) \\
        x_i &\stackrel{\text{iid}}{\sim} \text{Uniform}[0, 100]
    \end{aligned}
    $$

-   Want to approximate this function with polynomials

### Step 1: Create sample

In [ ]:
import numpy as np


def compute_true_y(x):
    """
    True trigonometric function (without errors)
    """
    return np.cos(3 * np.pi / 200.0 * x)

In [ ]:
from numpy.random import default_rng


def create_trig_sample(N=200, sigma=0.5, rng=None):
    """
    Create trigonometric relationship sample data for Ridge and Lasso.

    Parameters
    ----------
    N : int
        Sample size.
    sigma : float
        Standard deviation of the normal error term.
    rng : np.random.Generator, optional
        Random number generator.
    """

    # Initialize random number generator
    if rng is None:
        rng = default_rng(seed=1234)

    # Randomly draw explanatory variable x uniformly distributed on [0, 100]
    x = rng.uniform(0, 100, size=N)

    # Draw errors from normal distribution
    epsilon = rng.normal(scale=sigma, size=N)

    # Compute y, add measurement error
    y = compute_true_y(x) + epsilon

    return x, y

In [ ]:
# Sample size
N = 200

# Standard deviation of error term
sigma = 0.5

# Create sample data for trigonometric relationship between x and y
x, y = create_trig_sample(N=N, sigma=sigma)

### Step 2: Visualize sample and true function

In [ ]:
import matplotlib.pyplot as plt


def plot_trig_sample(x, y):
    """
    Plot the trigonometric relationship sample for Ridge and Lasso
    """
    # Sample scatter plot
    fig, ax = plt.subplots(1, 1, figsize=(5.5, 3.5))
    ax.scatter(x, y, s=20, c='none', edgecolor='steelblue', lw=0.75, label='Sample')

    # Plot true relationship
    xvalues = np.linspace(0.0, 100.0, 101)
    y_true = compute_true_y(xvalues)
    ax.plot(xvalues, y_true, c='black', lw=1.0, label='True function')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.legend(loc='upper right')

    return ax

In [ ]:
plot_trig_sample(x, y)

### Step 3: Estimate Ridge regression

1.  Assume the function is **approximated** by a polynomial of degree $K$:

    $$
    y_i \approx \mu + \beta_1 x_i + \beta_2 x_i^2 + \cdots + \beta_K x_i^K 
    $$

    -   Set $K=15$ for illustration

2.  Create pipeline ([`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) or [`make_pipeline()`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)):

    1.  Feature transformation: [`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)
    2.  Feature standardization: [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
    3.  Estimation: [`Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)
        -   Use regularization strength $\alpha = 3$
       
3.  Estimate model

In [ ]:
# Max. polynomial degree
degree = 15

# Penalty strength to use for Ridge
alpha = 3

# TODO: Build pipeline of transformations and Ridge regression
# pipe_ridge =

# TODO: Fit ridge regression

### Step 4: Estimate benchmark OLS regression

-   Useful as a benchmark model
-   Do we need feature standardization?

In [ ]:
# TODO: Create pipeline with linear regression
# pipe_lr =

### Step 5: Plot predicted values

In [ ]:
import numpy as np

# Values at which to predict
xvalues = np.linspace(0.0, 100.0, 100)

# TODO: Compute predicted values from Ridge
# y_pred_ridge =

# TODO: Compute predicted values from linear regression
# y_pred_lr =

In [ ]:
# Plot sample and true relationship
ax = plot_trig_sample(x, y)

# Linear regression prediction
ax.plot(xvalues, y_pred_lr, c='purple', alpha=0.7, label='Linear regression')

# Ridge prediction
ax.plot(xvalues, y_pred_ridge, c='darkorange', lw=2.0, label='Ridge')
ax.set_title(rf'Ridge regression with $\alpha$ = {alpha:.4g} vs OLS benchmark')
ax.legend()

<div style="padding: 0.8em 1em 0.5em 1em; border: 2pt solid #1d91c0;">
<p style="font-weight: bold; font-size: 1.5em; color: #1d91c0;">Your turn</p>

Rerun the above estimation for Ridge and OLS, but remove the `StandardScaler()` from the pipeline. What happens to the predicted values?
</div>
<span style="display: none;">YourTurnEnd</span>

***
## Intuition: coefficients vs. regularization strength

-   What happens to the magnitudes of the estimated coefficients as we vary regularization strength $\alpha$?
-   Fit many Ridge models over a grid of $\alpha$ and plot the coefficients
-   **Ridge path:** Visualization of how coefficients change with $\alpha$

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Create grid of alphas spaced uniformly in logs on [5e-3, 1000]
alphas = np.logspace(start=np.log10(5.0e-3), stop=np.log10(1000), num=100)

# Re-create pipeline w/o Ridge estimator, estimation step differs for each alpha
transform = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False), StandardScaler()
)

# Create polynomial features
X_trans = transform.fit_transform(x[:, None])

# Array to store coefficients for all alphas
coefs = np.empty((len(alphas), X_trans.shape[1]))

# TODO: loop over alphas, fit Ridge for each alpha
# TODO: Store coefficients from coef_ attribute

In [ ]:
import matplotlib.pyplot as plt

# Plot coefficient arrays against penalty strength
plt.figure(figsize=(6, 4))

plt.plot(alphas, coefs, lw=1.0)

plt.xscale('log', base=10)
plt.axhline(0.0, ls='--', lw=0.75, c='black')
plt.xlabel(r'Regularization strength $\alpha$ (log scale)')
plt.ylabel('Coefficient value')
plt.title('Ridge coefficients as function of regularization strength')
_ = plt.legend([rf'$\beta_{{{i}}}$' for i in range(degree)], ncols=5, loc='lower right')

***
## Tuning the regularization parameter via cross-validation

-   Regularization strength $\alpha$ can be cross-validated with
    [`RidgeCV`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html)
-   Uses MSE to find the optimal $\alpha$
-   Use the argument `store_cv_results=True` to store the MSE for all candidate values of $\alpha$ (to plot the validation curve)

### Step 1: Run Ridge CV

-   `RidgeCV` does not support pipelines, so transform features manually

In [ ]:
from sklearn.linear_model import RidgeCV

# RidgeCV does not support pipelines, so we need to transform x before
# cross-validation.
transform = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False), StandardScaler()
)

# Create standardized polynomial features
X_trans = transform.fit_transform(x[:, None])

# Candidate alphas used for cross-validation, spaced uniformly in logs to get
# denser grid for small alphas.
N_alphas = 100
alphas = np.logspace(start=np.log10(1.0e-5), stop=np.log10(5), num=N_alphas)

# TODO: fit RidgeCV with alphas (use store_cv_results=True to get MSE for each alpha)
# rcv =

# TODO: store and report best alpha (alpha_ attribute)
# TODO: store and report MSE (best_score_ attribute)

### Step 2: Plot validation curve

In [ ]:
# TODO: Compute average MSE for each alpha
# mse_mean =

In [ ]:
import matplotlib.pyplot as plt

# Plot MSE against alphas, highlight minimum MSE
plt.plot(alphas, mse_mean)
plt.xlabel(r'Regularization strength $\alpha$ (log scale)')
plt.ylabel('Cross-validated MSE')
plt.scatter(alphas[imin], mse_mean[imin], s=15, c='black', zorder=100)
plt.axvline(alphas[imin], ls=':', lw=0.75, c='black')
plt.xscale('log')
_ = plt.title('Validation curve for Ridge regression')

### Step 3: Re-estimate model with optimal $\alpha$ (optional)

-   Not strictly needed; could directly use the fitted `RidgeCV` object
-   But `RidgeCV` does not support pipelines...

In [ ]:
# TODO: Create pipeline with optimal alpha
pipe_ridge = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False),
    StandardScaler(),
    # TODO: Add Ridge estimator
)

# TODO: Fit Ridge with optimal alpha

### Step 4: Plot predictions from optimal Ridge

In [ ]:
# Grid on which to evaluate predictions
xvalues = np.linspace(np.amin(x), np.amax(x), 100)

# TODO: Predicted values from Ridge regression
# y_pred =

In [ ]:
# Plot sample and true relationship
ax = plot_trig_sample(x, y)

# Plot predicted values from cross-validated Ridge regression
ax.plot(xvalues, y_pred, c='darkorange', lw=2.0, label=r'Ridge (optimal $\alpha$)')
ax.set_title(rf'Ridge regression with optimal $\alpha$ = {alpha_best:.4g}')
ax.legend()

<div style="padding: 0.8em 1em 0.5em 1em; border: 2pt solid #1d91c0;">
<p style="font-weight: bold; font-size: 1.5em; color: #1d91c0;">Your turn</p>

Rerun the whole Ridge example with a smaller sample size of $N=50$. What happens to the optimal cross-validated penalty parameter $\alpha$?
</div>
<span style="display: none;">YourTurnEnd</span>

***
# Lasso

-   Same idea as Ridge regression, but with a different penalty term:
    $$
    L(\mu, \bm{\beta}) = 
        \frac{1}{2N} \underbrace{\sum_{i=1}^N \Bigl(
        y_i - \mu - \mathbf{x}_i'\bm{\beta}\Bigr)^2}_{\text{Sum of squared errors}}
        + 
        \underbrace{\alpha \sum_{k=1}^K |\beta_k|}_{\text{L1 penalty}}
    $$

-   L1 penalty leads to **sparse models** with **fewer** nonzero coefficients

## Fitting the polynomial model with Lasso

### Step 1: Create sample

-   Recreate the same sample as in the Ridge example

In [ ]:
# Sample size
N = 200

# Standard deviation of error term
sigma = 0.5

# Create sample data for trigonometric relationship between x and y
x, y = create_trig_sample(N=N, sigma=sigma)

### Step 2: Estimate Lasso

1.  Create pipeline ([`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) or [`make_pipeline()`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)):

    1.  Feature transformation: [`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)
    2.  Feature standardization: [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
    3.  Estimation: [`Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)
        -   Use regularization strength $\alpha = 0.0075$
        -   Might need to increase the `max_iter` argument
       
2.  Estimate model


In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

# Polynomial degree
degree = 15

# Penalty strength to use for Lasso
alpha = 0.0075

# TODO: Build pipeline of transformations and Lasso estimation.
pipe_lasso = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False),
    StandardScaler(),
    # TODO: Add Lasso estimator
)

# TODO: Fit Lasso

### Step 3: Plot predicted values

In [ ]:
# Grid on which to evaluate predictions
xvalues = np.linspace(np.amin(x), np.amax(x), 100)

# TODO: Predicted values from Lasso regression
# y_pred_lasso =

In [ ]:
ax = plot_trig_sample(x, y)

# Linear regression prediction
ax.plot(xvalues, y_pred_lr, c='purple', alpha=0.7, label='Linear regression')

# Lasso prediction
ax.plot(xvalues, y_pred_lasso, c='darkorange', lw=2.0, label='Lasso')

ax.set_title(rf'Lasso with $\alpha$ = {alpha:.4g} vs OLS benchmark')
ax.legend()

***
## Intuition: coefficients vs. regularization strength

-   **Lasso path**: visualizes how coefficients change as a function of the penalty $\alpha$
-   Use [`lasso_path()`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.lasso_path.html)
    to compute coefficients for a grid of $\alpha$

In [ ]:
from sklearn.linear_model import lasso_path

# Create grid of alphas spaced uniformly in logs on [1e-3, 1]
alphas = np.logspace(start=np.log10(1.0e-3), stop=np.log10(1.0), num=100)

# Re-create pipeline w/o Lasso estimator, estimation step differs for each alpha
transform = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False), StandardScaler()
)

# Create polynomial features
X_trans = transform.fit_transform(x[:, None])

# Compute Lasso path, store alphas, coefs

#### Plot coefficient magnitudes on an $\alpha$ grid

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(alphas, coefs.T, lw=1.0)
plt.xscale('log', base=10)
plt.axhline(0.0, ls='--', lw=0.75, c='black')
plt.xlabel(r'Regularization strength $\alpha$ (log scale)')
plt.ylabel('Coefficient value')
plt.title('Lasso coefficients as function of regularization strength')
_ = plt.legend([rf'$\beta_{{{i}}}$' for i in range(degree)], ncols=5)

#### Plot number of nonzero coefficients against the $\alpha$ grid

In [ ]:
# Number of nonzero coefficients for each alpha.
nonzero = np.sum(np.abs(coefs) > 1.0e-6, axis=0).astype(int)

# Plot number of nonzero coefficients against alpha
plt.plot(alphas, nonzero, lw=1.5, c='steelblue')
plt.xscale('log', base=10)
plt.yticks(np.arange(0, np.amax(nonzero) + 1))
plt.xlabel(r'Regularization strength $\alpha$ (log scale)')
_ = plt.title('Number of nonzero coefficients')

***
## Tuning the regularization parameter via cross-validation

-   Regularization strength $\alpha$ can be cross-validated with
    [`LassoCV`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html)
-   Uses MSE to find the optimal $\alpha$
-   Instead of specifying the $\alpha$ grid, we can specify $\epsilon = \frac{\alpha_{min}}{\alpha_{max}}$ (default: $10^{-3}$)
    and the desired number of points for the grid of $\alpha$ values

### Step 1: Run Lasso CV

-   `LassoCV` does not support pipelines, so transform features manually

In [ ]:
# LassoCV does not support pipelines, so we need to transform x before
# cross-validation.
transform = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False), StandardScaler()
)

# Create standardized polynomial features
X_trans = transform.fit_transform(x[:, None])

# TODO: Create and run Lasso cross-validation, use defaults for eps and n_alphas
# lcv =

# TODO: Store and report best alpha
# alpha_best =

# TODO: Report number of nonzero coefficients

### Step 2: Plot validation curve

- Visualize how MSE changes with the regularization strength

In [ ]:
# TODO: Compute average MSE for each alpha
# mse_mean =

# TODO: Compute index of minimal MSE
# imin =

In [ ]:
import matplotlib.pyplot as plt

# Recover grid of alphas used for CV
alphas = lcv.alphas_

# Plot MSE against alphas, highlight minimum MSE
plt.plot(alphas, mse_mean)
plt.xlabel(r'Regularization strength $\alpha$ (log scale)')
plt.ylabel('Cross-validated MSE')
plt.scatter(alphas[imin], mse_mean[imin], s=15, c='black', zorder=100)
plt.axvline(alphas[imin], ls=':', lw=0.75, c='black')
plt.xscale('log')
_ = plt.title('Validation curve for Lasso')

### Step 3: Re-estimate model with optimal $\alpha$ (optional)

-   Not strictly needed; could directly use the fitted `LassoCV` object
-   But `LassoCV` does not support pipelines...

In [ ]:
# TODO: Create pipeline with Lasso using optimal alpha
pipe_lasso = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False),
    StandardScaler(),
    # TODO: Add Lasso estimator with optimal alpha
)

# TODO: Fit Lasso with optimal alpha

### Step 4: Plot predicted values from optimal model

In [ ]:
# Grid on which to evaluate predictions
xvalues = np.linspace(np.amin(x), np.amax(x), 100)

# TODO: Predicted values from Lasso regression
# y_pred =

In [ ]:
# Plot sample and true relationship
ax = plot_trig_sample(x, y)

# Plot prediction from optimal Lasso model
plt.plot(xvalues, y_pred, c='darkorange', lw=2.0, label=r'Lasso (optimal $\alpha$)')

ax.set_title(rf'Lasso with optimal $\alpha$ = {alpha_best:.4g}')
_ = plt.legend()